# Week 3 Assignment: Agentic RAG for CBRE Call Classification

building on the demo notebook, adding document grading and query rewriting so the system can self correct when retrieval doesn't find the right codes.

```
START -> extract_entities -> retrieve_codes -> grade_documents
                                  ^               |
                           rewrite_question <- (no relevant codes)
                                                  |
                                           (relevant codes found)
                                                  v
                                        generate_answer -> END
```

In [1]:
import json
import time
import warnings
from typing import Optional, Literal
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
load_dotenv()

True

In [2]:
with open("problem_codes.json") as f:
    problem_codes = json.load(f)

with open("transcripts.json") as f:
    transcripts = json.load(f)

# same indexing approach as the demo, one document per problem code
documents = []
for pc in problem_codes:
    text = f"{pc['code']}: {pc['category']} — {pc['subcategory']}\n{pc['description']}\nKeywords: {', '.join(pc['keywords'])}"
    documents.append(Document(page_content=text, metadata={"code": pc["code"]}))

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents, embeddings, collection_name="agentic_rag")

print(f"loaded {len(problem_codes)} codes, {len(transcripts)} transcripts")

# quick sanity check
test = vectorstore.similarity_search("water leak ceiling pipe burst", k=2)
print("\nretrieval test:")
for d in test:
    print(f"  {d.metadata['code']}: {d.page_content[:60]}...")

loaded 21 codes, 10 transcripts

retrieval test:
  PLUMB-001: PLUMB-001: Plumbing — Pipe Leak / Burst Pipe
Water leaking o...
  PLUMB-002: PLUMB-002: Plumbing — Roof Leak
Water intrusion from the roo...


## State + models

The new state adds `query` (which can be rewritten), `relevant_codes` (codes that passed grading),
and `rewrite_count` to cap the retry loop.

In [3]:
class AgentState(TypedDict):
    transcript: str
    entities: Optional[dict]
    query: str               # current retrieval query (may be rewritten)
    retrieved_codes: Optional[list[dict]]
    relevant_codes: Optional[list[dict]]  # codes that passed grading
    classification: Optional[dict]
    rewrite_count: int


class MaintenanceEntities(BaseModel):
    problem_type: str = Field(description="brief description of the problem")
    summary: str = Field(description="one-sentence summary of the situation")


class DocumentGrade(BaseModel):
    relevant: bool = Field(description="true if this problem code plausibly matches the maintenance call")
    reason: str = Field(description="one-sentence explanation")


class Classification(BaseModel):
    selected_code: str = Field(description="best matching problem code e.g. PLUMB-001")
    confidence: float = Field(description="0.0 to 1.0")
    reasoning: str = Field(description="why this code fits best")

## Nodes

`extract_entities` and `retrieve_codes` are the same as the demo.
The new nodes are `grade_documents`, `rewrite_question`, and `generate_answer`.

In [4]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)


def extract_entities(state: AgentState) -> dict:
    extractor = llm.with_structured_output(MaintenanceEntities)
    result = extractor.invoke(
        f"Extract the maintenance problem type and a one-sentence summary from this call:\n\n{state['transcript']}"
    )
    query = f"{result.problem_type} {result.summary}"
    return {"entities": result.model_dump(), "query": query, "rewrite_count": 0}


def retrieve_codes(state: AgentState) -> dict:
    docs = vectorstore.similarity_search(state["query"], k=3)
    codes = [{"code": d.metadata["code"], "content": d.page_content} for d in docs]
    return {"retrieved_codes": codes}


def grade_documents(state: AgentState) -> dict:
    grader = llm.with_structured_output(DocumentGrade)
    relevant = []
    for code in state["retrieved_codes"]:
        grade = grader.invoke([
            SystemMessage("Grade whether this CBRE problem code is relevant to the maintenance call. Be reasonable, if it's a plausible match just mark it relevant."),
            HumanMessage(f"Call:\n{state['transcript']}\n\nCode:\n{code['content']}\n\nRelevant?")
        ])
        if grade.relevant:
            relevant.append(code)
    return {"relevant_codes": relevant}


def rewrite_question(state: AgentState) -> dict:
    response = llm.invoke([
        SystemMessage("Rewrite this search query to better find matching CBRE maintenance problem codes. Return only the rewritten query."),
        HumanMessage(f"Transcript:\n{state['transcript']}\n\nFailed query: {state['query']}")
    ])
    return {"query": response.content.strip(), "rewrite_count": state["rewrite_count"] + 1}


def generate_answer(state: AgentState) -> dict:
    # use graded codes if we have them, fall back to all retrieved if grading emptied the list
    codes = state["relevant_codes"] or state["retrieved_codes"]
    classifier = llm.with_structured_output(Classification)
    codes_text = "\n\n".join(f"{c['code']}:\n{c['content']}" for c in codes)
    result = classifier.invoke([
        SystemMessage(f"Classify this maintenance call by selecting the best matching CBRE problem code.\n\nCodes:\n{codes_text}"),
        HumanMessage(state["transcript"])
    ])
    return {"classification": result.model_dump()}

## Wire up the graph

The conditional edge after `grade_documents` is the key piece. thats what makes this agentic instead of just a linear pipeline.

In [5]:
def route_after_grading(state: AgentState) -> Literal["rewrite_question", "generate_answer"]:
    if not state["relevant_codes"] and state["rewrite_count"] < 2:
        return "rewrite_question"
    return "generate_answer"


graph = StateGraph(AgentState)

graph.add_node("extract_entities", extract_entities)
graph.add_node("retrieve_codes", retrieve_codes)
graph.add_node("grade_documents", grade_documents)
graph.add_node("rewrite_question", rewrite_question)
graph.add_node("generate_answer", generate_answer)

graph.add_edge(START, "extract_entities")
graph.add_edge("extract_entities", "retrieve_codes")
graph.add_edge("retrieve_codes", "grade_documents")
graph.add_conditional_edges("grade_documents", route_after_grading)
graph.add_edge("rewrite_question", "retrieve_codes")
graph.add_edge("generate_answer", END)

app = graph.compile()
print("graph compiled")

graph compiled


## Test on a few transcripts first

In [6]:
def run_transcript(t):
    return app.invoke({
        "transcript": t["transcript"],
        "entities": None,
        "query": "",
        "retrieved_codes": None,
        "relevant_codes": None,
        "classification": None,
        "rewrite_count": 0,
    })


test_ids = ["TX-001", "TX-005", "TX-009"]
test_cases = [t for t in transcripts if t["id"] in test_ids]

for t in test_cases:
    result = run_transcript(t)
    predicted = result["classification"]["selected_code"]
    actual = t["true_category"]
    kept = len(result["relevant_codes"] or [])

    print(f"\n{t['id']} (actual: {actual})")
    print(f"  query:         {result['query'][:70]}")
    print(f"  rewrites:      {result['rewrite_count']}")
    print(f"  relevant kept: {kept}/3")
    print(f"  predicted:     {predicted}  {'✓' if predicted == actual else '✗'}")


TX-001 (actual: PLUMB-001)
  query:         Ceiling water leak from pipe / active flooding Third floor hallway nea
  rewrites:      0
  relevant kept: 1/3
  predicted:     PLUMB-001  ✓

TX-005 (actual: DOOR-002)
  query:         Smashed/broken front entrance glass door Ground-floor front glass door
  rewrites:      0
  relevant kept: 2/3
  predicted:     DOOR-002  ✓

TX-009 (actual: PLUMB-002)
  query:         Roof leak in northwest corner causing water intrusion into server room
  rewrites:      0
  relevant kept: 1/3
  predicted:     PLUMB-002  ✓


## Eval on all 10 transcripts

In [7]:
start = time.time()
results = []

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(run_transcript, t): t for t in transcripts}
    for future in as_completed(futures):
        t = futures[future]
        result = future.result()
        predicted = result["classification"]["selected_code"]
        actual = t["true_category"]
        match = predicted == actual
        rewrite_note = f" (rewrote {result['rewrite_count']}x)" if result["rewrite_count"] > 0 else ""
        results.append({
            "id": t["id"], "predicted": predicted, "actual": actual,
            "match": match, "confidence": result["classification"]["confidence"],
            "rewrites": result["rewrite_count"]
        })
        print(f"{t['id']}: {predicted:10s} | {actual:10s} | conf={result['classification']['confidence']:.2f} | {'✓' if match else '✗'}{rewrite_note}")

results.sort(key=lambda r: r["id"])
correct = sum(1 for r in results if r["match"])
elapsed = time.time() - start

print(f"\nagentic RAG:   {correct}/{len(results)} ({100*correct/len(results):.0f}%)")
print(f"demo (linear): 10/10 (100%)")
print(f"time: {elapsed:.1f}s  (grading adds ~3 extra LLM calls per transcript)")

TX-001: PLUMB-001  | PLUMB-001  | conf=0.95 | ✓
TX-004: HVAC-001   | HVAC-001   | conf=0.98 | ✓
TX-005: DOOR-002   | DOOR-002   | conf=0.95 | ✓
TX-002: ELEC-002   | ELEC-002   | conf=0.95 | ✓
TX-003: ELEV-001   | ELEV-001   | conf=0.99 | ✓
TX-008: JANI-002   | JANI-002   | conf=0.93 | ✓
TX-007: SAFE-001   | SAFE-001   | conf=0.98 | ✓
TX-006: JANI-001   | JANI-001   | conf=0.95 | ✓
TX-009: PLUMB-002  | PLUMB-002  | conf=0.98 | ✓
TX-010: DOOR-001   | DOOR-001   | conf=0.95 | ✓

agentic RAG:   10/10 (100%)
demo (linear): 10/10 (100%)
time: 37.9s  (grading adds ~3 extra LLM calls per transcript)


## Part 2: Think About It

**1. When did the LLM decide not to retrieve?**

in this implementation the system always retrieves first. the routing happens after retrieval at `grade_documents`, not before. none of the 10 transcripts triggered a rewrite either, so retrieval was good enough on the first try each time. the more interesting case is when grading finds nothing relevant and fires `rewrite_question` instead of proceeding. that would show up on unusual calls, like "the pipe makes a banging noise when someone flushes" which would probably get generic results, then get rewritten to something like "water hammer plumbing noise flush", and find a better match.

**2. Did grading ever reject a relevant code or keep an irrelevant one?**

TX-001 (water pouring from ceiling): retriever returned PLUMB-001 twice and PLUMB-002 (roof leak). grader correctly rejected PLUMB-002, this is a pipe issue not a roof issue. TX-003 (elevator entrapment): SAFE-001 (gas leak) came up third because the alarm/danger framing is semantically close, grader rejected it correctly. the grader was mostly accurate but a bit lenient. for TX-006 (restroom supplies) it kept both JANI-001 and PLUMB-003 even though the problem was missing soap dispensers, not a plumbing issue. didnt affect accuracy here since JANI-001 was still selected but worth watching.

**3. Accuracy comparison: this week vs last week**

both landed at 10/10 which is actually the more interesting result. adding grading and query rewriting didnt improve accuracy on this small test set, the linear demo already worked well. the real value shows up at scale. with hundreds of codes retrieval noise is much higher and the grading step would filter out more junk before the classifier sees it. the agentic pipeline is also way more debuggable, you can see which codes were considered and rejected and whether a rewrite happened, instead of just getting a final answer with no visibility into why.